# 제작비-관객 수 기반 영화 유형 군집화

KMeans로 제작비와 누적 관객 수의 상대적 위치를 나눕니다. TMDB 제작비는 결측과 0값이 많고 통화 기준이 완전히 통일되어 있지 않을 수 있어, 군집은 절대 BEP가 아니라 상대적 포지셔닝으로 해석합니다.

In [ ]:
from pathlib import Path
import re
import warnings

import matplotlib.font_manager as fm
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore", category=UserWarning)
RANDOM_STATE = 42


def find_project_root() -> Path:
    current = Path.cwd().resolve()
    for path in [current, *current.parents]:
        if (path / "data").exists() and (path / "notebooks").exists():
            return path
    return current.parent if current.name == "notebooks" else current


ROOT = find_project_root()


def configure_korean_font() -> None:
    preferred_fonts = ["AppleGothic", "NanumGothic", "NanumBarunGothic", "Malgun Gothic"]
    available_fonts = {font.name for font in fm.fontManager.ttflist}
    for family in preferred_fonts:
        if family in available_fonts:
            plt.rcParams["font.family"] = family
            break
    plt.rcParams["axes.unicode_minus"] = False


configure_korean_font()
ROOT

In [ ]:
DATA_CANDIDATES = [
    ROOT / "data/processed/movie_features.csv",
    ROOT / "data/kobis_tmdb_title_matches.csv",
    Path("/content/drive/MyDrive/Colab Notebooks/top_500_korean_movies.csv"),
]
TITLE_COLUMNS = ["match_title", "title", "movie_name", "Movie Name"]
AUDIENCE_COLUMNS = ["audience_count", "kobis_audi_acc", "Admissions"]
BUDGET_COLUMNS = ["meta_tmdb_budget", "tmdb_budget", "budget", "Estimated Production Budget", "Budget(100M_KRW)"]


def read_first_available_csv(candidates):
    for path in candidates:
        if path.exists():
            return pd.read_csv(path, encoding="utf-8-sig"), path
    searched = "\n".join(str(path) for path in candidates)
    raise FileNotFoundError(f"분석용 CSV를 찾지 못했습니다. 확인 경로:\n{searched}")


def first_existing_column(frame: pd.DataFrame, candidates):
    for column in candidates:
        if column in frame.columns:
            return column
    raise ValueError(f"필수 컬럼이 없습니다. 후보: {candidates}")


def parse_budget(value: object) -> float:
    if pd.isna(value):
        return np.nan
    if isinstance(value, (int, float, np.integer, np.floating)):
        return float(value)
    text = str(value).replace(",", "").strip()
    if not text:
        return np.nan
    numeric = pd.to_numeric(text, errors="coerce")
    if pd.notna(numeric):
        return float(numeric)
    numbers = re.findall(r"\d+(?:\.\d+)?", text)
    return float(numbers[0]) if numbers else np.nan


raw_data, data_path = read_first_available_csv(DATA_CANDIDATES)
title_col = first_existing_column(raw_data, TITLE_COLUMNS)
audience_col = first_existing_column(raw_data, AUDIENCE_COLUMNS)
budget_col = first_existing_column(raw_data, BUDGET_COLUMNS)

movies = pd.DataFrame({
    "movie_title": raw_data[title_col],
    "audience_count": pd.to_numeric(raw_data[audience_col], errors="coerce"),
    "budget": raw_data[budget_col].map(parse_budget),
})
movies = movies.dropna(subset=["audience_count", "budget"])
movies = movies[(movies["audience_count"] > 0) & (movies["budget"] > 0)].copy()
movies["audience_million"] = movies["audience_count"] / 1_000_000
movies["budget_million"] = movies["budget"] / 1_000_000
movies["log_audience"] = np.log1p(movies["audience_count"])
movies["log_budget"] = np.log1p(movies["budget"])
movies["audience_per_budget_million"] = movies["audience_million"] / movies["budget_million"]

print(f"데이터: {data_path}")
print(f"제작비/관객 수가 모두 있는 영화 수: {len(movies):,}건")
movies[["movie_title", "budget", "audience_count", "audience_per_budget_million"]].head()

In [ ]:
if len(movies) < 8:
    raise ValueError("군집화를 수행하기에는 제작비가 있는 영화가 너무 적습니다.")

features = movies[["log_budget", "log_audience"]]
scaler = StandardScaler()
scaled = scaler.fit_transform(features)

max_k = min(7, len(movies) - 1)
silhouette_rows = []
for k in range(2, max_k + 1):
    labels = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=20).fit_predict(scaled)
    silhouette_rows.append({"k": k, "silhouette": silhouette_score(scaled, labels)})

silhouette_table = pd.DataFrame(silhouette_rows)
SELECTED_K = 4 if len(movies) >= 4 else int(silhouette_table.sort_values("silhouette", ascending=False).iloc[0]["k"])

kmeans = KMeans(n_clusters=SELECTED_K, random_state=RANDOM_STATE, n_init=20)
movies["cluster"] = kmeans.fit_predict(scaled)
silhouette_table

In [ ]:
summary = movies.groupby("cluster").agg(
    n_movies=("movie_title", "count"),
    median_budget_million=("budget_million", "median"),
    median_audience_million=("audience_million", "median"),
    mean_audience_million=("audience_million", "mean"),
    hit_rate_3m=("audience_count", lambda values: (values >= 3_000_000).mean()),
    median_audience_per_budget_million=("audience_per_budget_million", "median"),
)

overall_budget = movies["budget_million"].median()
overall_audience = movies["audience_million"].median()


def cluster_label(row: pd.Series) -> str:
    budget_high = row["median_budget_million"] >= overall_budget
    audience_high = row["median_audience_million"] >= overall_audience
    if budget_high and audience_high:
        return "대작 흥행형"
    if budget_high and not audience_high:
        return "고비용 저효율형"
    if not budget_high and audience_high:
        return "저비용 고효율형"
    return "소형 저관객형"

summary["cluster_label"] = summary.apply(cluster_label, axis=1)
movies["cluster_label"] = movies["cluster"].map(summary["cluster_label"])
summary.sort_values("median_audience_million", ascending=False)

In [ ]:
fig, ax = plt.subplots(figsize=(11, 7))
for label, part in movies.groupby("cluster_label"):
    ax.scatter(
        part["budget_million"],
        part["audience_million"],
        s=70,
        alpha=0.75,
        label=f"{label} ({len(part)})",
    )

for _, row in movies.nlargest(25, "audience_count").iterrows():
    ax.annotate(
        row["movie_title"],
        (row["budget_million"], row["audience_million"]),
        textcoords="offset points",
        xytext=(4, 4),
        fontsize=8,
    )

ax.axhline(3, color="#555555", linestyle="--", linewidth=1, label="300만 기준")
ax.set_xscale("log")
ax.set_yscale("log")
ax.set_title("제작비-관객 수 기준 영화 군집")
ax.set_xlabel("TMDB 제작비 (백만 단위, 로그축)")
ax.set_ylabel("누적 관객 수 (백만 명, 로그축)")
ax.grid(True, which="both", alpha=0.2)
ax.legend(loc="best")
plt.tight_layout()

In [ ]:
efficiency_columns = [
    "movie_title", "cluster_label", "budget_million", "audience_million", "audience_per_budget_million",
]
movies.sort_values(
    ["audience_per_budget_million", "audience_count"],
    ascending=False,
)[efficiency_columns].head(15)

## 해석 메모

- 제작비가 0이거나 비어 있는 영화는 군집에서 제외했습니다. 이 때문에 전체 영화 대표성은 제한됩니다.
- `저비용 고효율형`은 예산 대비 관객 성과가 높은 후보군입니다. 후속 분석에서는 장르, 감독, 개봉 월, 포스터 특성을 붙여 어떤 요인이 효율 차이를 만드는지 확인하는 것이 좋습니다.
- 실제 손익분기점 분석에는 제작비 통화, 마케팅비, 극장/배급 수익 배분율이 추가로 필요합니다.